In [1]:
# Environment: scMEDAL
import glob
import anndata
import sys
from pathlib import Path
ROOT_PATH  = Path.cwd().resolve().parents[2]  # two levels up from AML dir (scMEDAL_for_scRNAseq)
sys.path.insert(0, str(ROOT_PATH ))
print(ROOT_PATH )
from utils.splitter import DataSplitter
from utils.utils import  read_adata
from utils.model_train_utils import load_data
from config_split_paths import data_path,data2split_foldername,folder_splits
import numpy as np
import os

/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq


2025-10-06 16:18:10.533241: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 16:18:12.877221: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq
OUTPUTS_DIR: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq/outputs


In [2]:

# Read path
adata2split_path = os.path.join(data_path,folder_splits)
print(adata2split_path)
splits_list = glob.glob(adata2split_path+"/*")
glob.glob(splits_list[0]+"/train/*")

/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq/data/ASD_data/log_transformed_2916hvggenes/splits


['/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq/data/ASD_data/log_transformed_2916hvggenes/splits/split_1/train/exprMatrix.npy',
 '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq/data/ASD_data/log_transformed_2916hvggenes/splits/split_1/train/meta.csv',
 '/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq/data/ASD_data/log_transformed_2916hvggenes/splits/split_1/train/geneids.csv']

In [3]:
# get original donor data
adata2split_path_original = data_path + "/" + data2split_foldername
# X,var,obs = read_adata(adata2split_path_original)
X,var,obs = read_adata(adata2split_path_original, issparse=False)
adata = anndata.AnnData(X,obs=obs,var=var)

Reading data from: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/scMEDAL_for_scRNAseq/Experiments/ASD/../data/ASD_data/log_transformed_2916hvggenes


/project/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_ARMED_2/lib/python3.8/site-packages/anndata/_core/anndata.py:119: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [4]:
import numpy as np
len(np.unique(adata.obs["celltype"]))

17

In [4]:
# Creating a dict with split paths and adata_splits
def get_splits_dict(splits_path,issparse=False):
    splits_list = glob.glob(splits_path+"/*")
    #print(splits_list)
    split_paths_dict = {}
    adata_splits_dict = {}
    for split in splits_list:
        split_name = split.split("/")[-1]
        split_paths_dict[split_name]={"train":split+"/train","val":split+"/val","test":split+"/test"}
        print(split_paths_dict[split_name])
        adata_splits_dict[split_name]=load_data(split_paths_dict[split_name], eval_test=True, scaling=None,issparse=issparse)
    return split_paths_dict,adata_splits_dict

In [5]:
def check_fold_leakage(adata_splits_dict):
    val_indices = []
    test_indices = []

    # Extracting indices for validation and test sets from each fold in the dictionary
    for split_key in adata_splits_dict.keys():
        fold_data = adata_splits_dict[split_key]
        val_indices.append(set(fold_data['val'].obs['original_index']))
        test_indices.append(set(fold_data['test'].obs['original_index']))

    # Checking for any overlaps (leakage) in validation and test sets
    num_folds = len(adata_splits_dict)
    overlap_detected = False
    for i in range(num_folds):
        for j in range(i + 1, num_folds):
            val_intersection = val_indices[i].intersection(val_indices[j])
            test_intersection = test_indices[i].intersection(test_indices[j])
            if val_intersection:
                print(f"Overlap detected in validation sets between folds {i} and {j}: {len(val_intersection)} common elements")
                overlap_detected = True
            if test_intersection:
                print(f"Overlap detected in test sets between folds {i} and {j}: {len(test_intersection)} common elements")
                overlap_detected = True

    if overlap_detected:
        return False
    else:
        print("No overlap detected in validation and test sets between folds.")
        return True


In [6]:
# Creating a dict with split paths and adata_splits
split_paths_dict,adata_splits_dict = get_splits_dict(data_path+"/"+folder_splits,issparse=True) 

{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_norm/log_transformed_2916hvggenes/splits/split_1/train', 'val': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_norm/log_transformed_2916hvggenes/splits/split_1/val', 'test': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_norm/log_transformed_2916hvggenes/splits/split_1/test'}
X.shape before scaling (62735, 2916)
X.shape before scaling (20912, 2916)
X.shape before scaling (20912, 2916)
{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_norm/log_transformed_2916hvggenes/splits/split_2/train', 'val': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_norm/log_transformed_2916hvggenes/splits/split_2/val', 'test': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_norm/log_transformed_2916hvggenes/splits/split_2/test'}
X.shape before scaling (62735, 2916)
X.shape before scaling (2091

In [7]:
data_path+folder_splits

'/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/ASD/reverse_normlog_transformed_2916hvggenes/splits'

In [8]:


# Check if there is any overlap between 
overlap_check_result = check_fold_leakage(adata_splits_dict)

No overlap detected in validation and test sets between folds.


In [9]:
# get train data from split 1
split_to_check = "split_5"
adata_train = adata_splits_dict[split_to_check]["train"]
print("adata_train shape",adata_train.shape)
# read test data from split 1
adata_test = adata_splits_dict[split_to_check]["test"]
print("adata_test shape",adata_test.shape)
# read val data from split 1
adata_val = adata_splits_dict[split_to_check]["val"]
print("adata_val shape",adata_val.shape)

adata_train shape (62736, 2916)
adata_test shape (20911, 2916)
adata_val shape (20912, 2916)


In [10]:
adata_test.obs

,Unnamed: 0,Unnamed: 0.1,cell,cluster,sample,individual,region,age,sex,diagnosis,...,RNA Integrity Number,genes,UMIs,RNA mitochondr. percent,RNA ribosomal percent,celltype,donor,batch,n_genes,original_index
0,1862,1862,CATTCGCGTGCGGTAA-1_4341_BA24,Neu-NRGN-I,4341_BA24,4341,ACC,13,M,Control,...,7.5,1667,2863,4.820119,1.362208,Neu-NRGN-I,donor_4341,donor_4341,1667,1862
1,76569,76569,CTACATTTCGGCTACG-1_5893_BA24,Neu-mat,5893_BA24,5893,ACC,19,M,Control,...,8.4,1905,3225,0.527132,1.085271,Neu-mat,donor_5893,donor_5893,1905,76569
2,6152,6152,CTTTGCGAGGCACATG-1_4341_BA46,IN-VIP,4341_BA46,4341,PFC,13,M,Control,...,7.2,4362,9830,1.434384,0.600204,IN-VIP,donor_4341,donor_4341,4362,6152
3,21192,21192,TGAGCCGGTCCATCCT-1_5242_BA24,Oligodendrocytes,5242_BA24,5242,ACC,15,M,Control,...,7.9,666,912,0.438597,0.328947,Oligodendrocytes,donor_5242,donor_5242,666,21192
4,76423,76423,CGCTGGACACGGTGTC-1_5893_BA24,L2/3,5893_BA24,5893,ACC,19,M,Control,...,8.4,3018,5747,0.208805,1.305029,L2/3,donor_5893,donor_5893,3018,76423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20906,769,769,TGCCCTATCGTCTGCT-1_1823_BA24,L2/3,1823_BA24,1823,ACC,15,M,Control,...,7.0,2655,5864,0.204639,0.733288,L2/3,donor_1823,donor_1823,2655,769
20907,60263,60263,CGACCTTTCGAGAGCA-1_5565_BA9,IN-PV,5565_BA9,5565,PFC,12,M,ASD,...,6.9,6170,19815,3.613424,0.393641,IN-PV,donor_5565,donor_5565,6169,60263
20908,44131,44131,TAAGTGCTCTGCAGTA-1_5419_PFC,L2/3,5419_PFC,5419,PFC,19,F,ASD,...,7.3,3537,6853,1.269517,4.406829,L2/3,donor_5419,donor_5419,3537,44131
20909,87498,87498,TAGCCGGAGACTAGAT-1_5939_BA9,L2/3,5939_BA9,5939,PFC,21,M,ASD,...,7.4,3044,6754,0.843944,0.784720,L2/3,donor_5939,donor_5939,3044,87498


In [12]:
# check that train, test and val should sum up total data
adata_train.shape[0]+adata_val.shape[0]+adata_test.shape[0]

104559

In [14]:
import numpy as np
# # check all donors list
adata.obs['batch'] = adata.obs["batch"].astype('category')
donor_list = list(np.unique(adata.obs["batch"]))
# Seen donor list
donor_seen_list = list(np.unique(adata_train.obs["batch"]))
print(donor_seen_list)

['donor_1823', 'donor_4341', 'donor_4849', 'donor_4899', 'donor_5144', 'donor_5163', 'donor_5242', 'donor_5278', 'donor_5294', 'donor_5387', 'donor_5391', 'donor_5403', 'donor_5408', 'donor_5419', 'donor_5531', 'donor_5538', 'donor_5554', 'donor_5565', 'donor_5577', 'donor_5841', 'donor_5864', 'donor_5879', 'donor_5893', 'donor_5936', 'donor_5939', 'donor_5945', 'donor_5958', 'donor_5976', 'donor_5978', 'donor_6032', 'donor_6033']


In [15]:
# check if the stratification worked

# call the splitter stratification test
splitter = DataSplitter()
comp_df = splitter.check_stratification(adata, adata_train, adata_val, adata_test, ['batch', 'celltype'])
# add donor and celltype columns
comp_df["batch"] = [x.split("_")[0] for x in comp_df.index ]
comp_df["celltype"] = [x.split("_")[1] for x in comp_df.index ]


# Donor is in seen list
condition1 = comp_df["batch"].isin(donor_seen_list)
# Values equal to zero
condition2 = comp_df[['Train', 'Validation', 'Test']].eq(0).any(axis=1)
filtered_df = comp_df.loc[condition1 & condition2]
# We want to see the seen donors well distributed trough the train, test and validation datasets. No zero entries.


In [16]:
filtered_df

,Original,Train,Validation,Test,batch,celltype


In [17]:
adata_val

AnnData object with n_obs × n_vars = 20912 × 2916
    obs: 'Unnamed: 0', 'Unnamed: 0.1', 'cell', 'cluster', 'sample', 'individual', 'region', 'age', 'sex', 'diagnosis', 'Capbatch', 'Seqbatch', 'post-mortem interval (hours)', 'RNA Integrity Number', 'genes', 'UMIs', 'RNA mitochondr. percent', 'RNA ribosomal percent', 'celltype', 'donor', 'batch', 'n_genes', 'original_index', 'combined_stratify_col'
    var: 'Unnamed: 0'

In [18]:
adata_train.obs["combined_stratify_col"]

0                     donor_4341_OPC
1                     donor_4341_OPC
2                 donor_5864_Neu-mat
3        donor_5278_Oligodendrocytes
4                  donor_5893_AST-FB
                    ...             
62731    donor_5577_Oligodendrocytes
62732                donor_5577_L5/6
62733                  donor_5565_L4
62734    donor_4341_Oligodendrocytes
62735                donor_5144_L2/3
Name: combined_stratify_col, Length: 62736, dtype: object

In [19]:
adata_train.obs["combined_stratify_col"]

0                     donor_4341_OPC
1                     donor_4341_OPC
2                 donor_5864_Neu-mat
3        donor_5278_Oligodendrocytes
4                  donor_5893_AST-FB
                    ...             
62731    donor_5577_Oligodendrocytes
62732                donor_5577_L5/6
62733                  donor_5565_L4
62734    donor_4341_Oligodendrocytes
62735                donor_5144_L2/3
Name: combined_stratify_col, Length: 62736, dtype: object

In [20]:
# from matplotlib import pyplot as plt
# index_val = adata_val.obs.loc[adata_val.obs['combined_stratify_col']=='5163_Neu-mat'].index
# print("index_val.shape",index_val.shape)
# plt.hist(adata_val.X[np.array(index_val.astype(int)),:])


In [21]:
adata_train.obs

,Unnamed: 0,Unnamed: 0.1,cell,cluster,sample,individual,region,age,sex,diagnosis,...,genes,UMIs,RNA mitochondr. percent,RNA ribosomal percent,celltype,donor,batch,n_genes,original_index,combined_stratify_col
0,7696,7696,TGGCGCAGTCACTTCC-1_4341_BA46,OPC,4341_BA46,4341,PFC,13,M,Control,...,1739,3081,0.259656,0.714054,OPC,donor_4341,donor_4341,1739,7696,donor_4341_OPC
1,1441,1441,ATCCGAATCCGTTGTC-1_4341_BA24,OPC,4341_BA24,4341,ACC,13,M,Control,...,1252,1942,0.308960,0.514933,OPC,donor_4341,donor_4341,1252,1441,donor_4341_OPC
2,71644,71644,AGCATACAGGCCATAG-1_5864_BA9,Neu-mat,5864_BA9,5864,PFC,20,M,ASD,...,1901,3266,0.153092,1.132884,Neu-mat,donor_5864,donor_5864,1900,71644,donor_5864_Neu-mat
3,22374,22374,GACCAATAGCATCATC-1_5278_BA24,Oligodendrocytes,5278_BA24,5278,ACC,15,F,ASD,...,571,679,4.712813,2.061856,Oligodendrocytes,donor_5278,donor_5278,571,22374,donor_5278_Oligodendrocytes
4,76890,76890,GATCGTAGTTGAGGTG-1_5893_BA24,AST-FB,5893_BA24,5893,ACC,19,M,Control,...,1895,3338,0.179748,0.958658,AST-FB,donor_5893,donor_5893,1895,76890,donor_5893_AST-FB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62731,64925,64925,GTTCATTTCACTCTTA-1_5577_BA9,Oligodendrocytes,5577_BA9,5577,PFC,21,M,Control,...,1233,2105,0.095012,0.570071,Oligodendrocytes,donor_5577,donor_5577,1233,64925,donor_5577_Oligodendrocytes
62732,62955,62955,CATCAGAGTATATGAG-1_5577_BA9,L5/6,5577_BA9,5577,PFC,21,M,Control,...,5338,18159,0.319401,0.468087,L5/6,donor_5577,donor_5577,5338,62955,donor_5577_L5/6
62733,59735,59735,AGCGGTCGTAGAAGGA-1_5565_BA9,L4,5565_BA9,5565,PFC,12,M,ASD,...,3949,8937,0.481146,0.660177,L4,donor_5565,donor_5565,3949,59735,donor_5565_L4
62734,6265,6265,GACCTGGGTGAACCTT-1_4341_BA46,Oligodendrocytes,4341_BA46,4341,PFC,13,M,Control,...,715,1100,0.363636,1.181818,Oligodendrocytes,donor_4341,donor_4341,715,6265,donor_4341_Oligodendrocytes
